In [1]:
import pandas as pd
df = pd.read_csv("tickets.csv")
df.head()

,ticket_id,short_description,category,subcategory,issue_type
0,1,Unable to connect to VPN from home network,Network,VPN,Access Issue
1,2,Outlook not sending emails,Software,MS Outlook,Email Issue
2,3,Laptop screen flickering intermittently,Hardware,Display,Performance Issue
3,4,Password reset not working for AD account,Access Management,Active Directory,Access Issue
4,5,WiFi disconnects frequently in office,Network,WiFi,Connectivity Issue


In [3]:
!pip install nltk

                                              0.0/1.5 MB ? eta -:--:--
     -                                        0.0/1.5 MB 991.0 kB/s eta 0:00:02
     -----                                    0.2/1.5 MB 2.8 MB/s eta 0:00:01
     -----------                              0.4/1.5 MB 3.3 MB/s eta 0:00:01
     ---------------                          0.6/1.5 MB 3.4 MB/s eta 0:00:01
     ------------------                       0.7/1.5 MB 3.2 MB/s eta 0:00:01
     -------------------------                1.0/1.5 MB 3.6 MB/s eta 0:00:01
     -----------------------------            1.1/1.5 MB 3.6 MB/s eta 0:00:01
     -----------------------------------      1.3/1.5 MB 3.9 MB/s eta 0:00:01
     ---------------------------------------  1.5/1.5 MB 3.9 MB/s eta 0:00:01
     ---------------------------------------  1.5/1.5 MB 3.9 MB/s eta 0:00:01
     ---------------------------------------- 1.5/1.5 MB 3.0 MB/s eta 0:00:00



[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:

import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df['clean_text'] = df['short_description'].apply(clean_text)

df[['short_description', 'clean_text']].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Mansi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


,short_description,clean_text
0,Unable to connect to VPN from home network,unable connect vpn home network
1,Outlook not sending emails,outlook sending emails
2,Laptop screen flickering intermittently,laptop screen flickering intermittently
3,Password reset not working for AD account,password reset working ad account
4,WiFi disconnects frequently in office,wifi disconnects frequently office


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(df['clean_text'])
y = df['category']

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.3
              precision    recall  f1-score   support

    Hardware       1.00      0.25      0.40         4
     Network       0.12      1.00      0.22         1
    Software       1.00      0.20      0.33         5

    accuracy                           0.30        10
   macro avg       0.71      0.48      0.32        10
weighted avg       0.91      0.30      0.35        10



In [7]:
df['predicted_category'] = model.predict(X)

df['validation_flag'] = df.apply(
    lambda row: "Mismatch" if row['category'] != row['predicted_category'] else "Correct",
    axis=1
)

df[['ticket_id', 'category', 'predicted_category', 'validation_flag']].head()

,ticket_id,category,predicted_category,validation_flag
0,1,Network,Network,Correct
1,2,Software,Software,Correct
2,3,Hardware,Hardware,Correct
3,4,Access Management,Access Management,Correct
4,5,Network,Network,Correct


In [8]:
# Get prediction probabilities
probs = model.predict_proba(X)

# Get max probability for each prediction
import numpy as np
confidence_scores = np.max(probs, axis=1)

df['confidence_score'] = confidence_scores
df['predicted_category'] = model.predict(X)

In [9]:
def validate_ticket(row):
    if row['category'] != row['predicted_category']:
        return "Mismatch"
    elif row['confidence_score'] < 0.60:
        return "Low Confidence"
    else:
        return "Correct"

df['validation_flag'] = df.apply(validate_ticket, axis=1)

df[['ticket_id', 'category', 'predicted_category', 
    'confidence_score', 'validation_flag']].head()

,ticket_id,category,predicted_category,confidence_score,validation_flag
0,1,Network,Network,0.535979,Low Confidence
1,2,Software,Software,0.418502,Low Confidence
2,3,Hardware,Hardware,0.409062,Low Confidence
3,4,Access Management,Access Management,0.421267,Low Confidence
4,5,Network,Network,0.482304,Low Confidence


In [10]:
df['validation_flag'].value_counts()

validation_flag
Low Confidence    43
Mismatch           7
Name: count, dtype: int64

In [11]:
mismatch_percentage = (df['validation_flag'].value_counts(normalize=True) * 100)
print(mismatch_percentage)

validation_flag
Low Confidence    86.0
Mismatch          14.0
Name: proportion, dtype: float64


In [12]:
import pickle

# Save model
with open("ticket_classifier.pkl", "wb") as f:
    pickle.dump(model, f)

# Save vectorizer
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("Model and vectorizer saved successfully.")

Model and vectorizer saved successfully.


In [13]:
import numpy as np

feature_names = vectorizer.get_feature_names_out()

for i, category in enumerate(model.classes_):
    top10 = np.argsort(model.coef_[i])[-10:]
    print(f"\nTop words for {category}:")
    print([feature_names[j] for j in top10])


Top words for Access Management:
['employee', 'finance', 'revoked', 'incorrectly', 'user', 'account', 'multiple', 'new', 'access', 'need']

Top words for Hardware:
['heavy', 'usage', 'overheating', 'printing', 'documents', 'printer', 'running', 'turning', 'monitor', 'laptop']

Top words for Network:
['authentication', 'website', 'loading', 'desktop', 'frequently', 'network', 'internal', 'drive', 'vpn', 'internet']

Top words for Software:
['emails', 'sending', 'payroll', 'attachments', 'downloading', 'unable', 'software', 'file', 'crashing', 'update']


In [14]:
pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model_embed = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model_embed.encode(df['short_description'])
sim_matrix = cosine_similarity(embeddings)

In [16]:
!pip install streamlit

                                              0.0/9.1 MB ? eta -:--:--
                                              0.1/9.1 MB 2.0 MB/s eta 0:00:05
     -                                        0.3/9.1 MB 2.6 MB/s eta 0:00:04
     -                                        0.4/9.1 MB 2.6 MB/s eta 0:00:04
     --                                       0.5/9.1 MB 2.8 MB/s eta 0:00:04
     --                                       0.6/9.1 MB 2.7 MB/s eta 0:00:04
     ---                                      0.8/9.1 MB 2.7 MB/s eta 0:00:04
     ----                                     1.0/9.1 MB 2.9 MB/s eta 0:00:03
     ----                                     1.1/9.1 MB 3.0 MB/s eta 0:00:03
     -----                                    1.3/9.1 MB 3.1 MB/s eta 0:00:03
     ------                                   1.5/9.1 MB 3.1 MB/s eta 0:00:03
     -------                                  1.6/9.1 MB 3.1 MB/s eta 0:00:03
     -------                                  1.8/9.1 MB 3.2 MB/s eta 0


[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import streamlit as st
import pickle
import numpy as np

# Load model
model = pickle.load(open("ticket_classifier.pkl", "rb"))
vectorizer = pickle.load(open("tfidf_vectorizer.pkl", "rb"))

st.title("IT Ticket Validation System")

user_input = st.text_input("Enter Ticket Description")

if user_input:
    text_vec = vectorizer.transform([user_input])
    prediction = model.predict(text_vec)[0]
    confidence = np.max(model.predict_proba(text_vec))
    
    st.write("Predicted Category:", prediction)
    st.write("Confidence Score:", round(confidence, 2))
    
    if confidence < 0.6:
        st.warning("Low Confidence - Manual Review Suggested")

2026-03-02 17:30:44.510 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-02 17:30:45.003 
  command:

    streamlit run c:\Users\Mansi\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-03-02 17:30:45.004 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-02 17:30:45.005 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-02 17:30:45.006 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-02 17:30:45.007 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-02 17:30:45.008 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-02 17:30: